# Lab 3: Integration and Probability

The probabilistic quantities from the probability-foundations module — PDFs, expected values, entropy, KL divergence — are all integrals. This lab computes them numerically from scratch, verifies the Fundamental Theorem of Calculus, and shows why Monte Carlo sampling is how these integrals become tractable in high-dimensional ML.

**Sections**
1. The Trapezoidal Rule
2. Verifying the Fundamental Theorem
3. Gaussian Normalization and the 68–95–99.7 Rule
4. Monte Carlo Integration: Estimating π
5. KL Divergence Between Gaussians

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['figure.figsize'] = (9, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
rng = np.random.default_rng(0)
print('Setup complete.')

## 1. The Trapezoidal Rule

The trapezoidal rule approximates $\int_a^b f(x)\,dx$ by summing trapezoids of width $\Delta x$:
$$\int_a^b f(x)\,dx \approx \Delta x \left[\tfrac{1}{2}f(x_0) + f(x_1) + \cdots + f(x_{n-1}) + \tfrac{1}{2}f(x_n)\right]$$

Error scales as $O(1/n^2)$ — halving $\Delta x$ quarters the error.

In [ ]:
def trapz(f, a, b, n):
    """Trapezoidal rule using n subintervals."""
    x = np.linspace(a, b, n + 1)
    y = f(x)
    dx = (b - a) / n
    return dx * (0.5*y[0] + np.sum(y[1:-1]) + 0.5*y[-1])

# Test: ∫₀² x² dx = [x³/3]₀² = 8/3 ≈ 2.6667
f = lambda x: x**2
true_val = 8/3

n_vals = np.logspace(0.5, 4, 40).astype(int)
errors = [abs(trapz(f, 0, 2, n) - true_val) for n in n_vals]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Show the trapezoids visually
n_vis = 8
x_vis = np.linspace(0, 2, n_vis + 1)
x_fine = np.linspace(0, 2, 300)
ax1.plot(x_fine, f(x_fine), 'steelblue', lw=2, label='f(x)=x²')
for i in range(n_vis):
    ax1.fill_between([x_vis[i], x_vis[i+1]], [f(x_vis[i]), f(x_vis[i+1])],
                     alpha=0.3, color='coral')
    ax1.plot([x_vis[i], x_vis[i+1]], [f(x_vis[i]), f(x_vis[i+1])], 'coral', lw=0.8)
ax1.set_title(f'Trapezoidal rule (n={n_vis} trapezoids)')
ax1.legend()

# Error convergence
ax2.loglog(n_vals, errors, 'steelblue', lw=2, label='actual error')
ax2.loglog(n_vals, 1.0/n_vals**2, 'k--', lw=1, alpha=0.6, label='O(1/n²) reference')
ax2.set_xlabel('n (subintervals)')
ax2.set_ylabel('|error|')
ax2.set_title('Error vs number of subintervals')
ax2.legend()

plt.tight_layout()
plt.show()

print(f"n=1000: estimate={trapz(f,0,2,1000):.8f}, true={true_val:.8f}")

## 2. Verifying the Fundamental Theorem

The FTC says $\int_a^b f(x)\,dx = F(b) - F(a)$ where $F' = f$. We verify this numerically for three functions.

In [ ]:
# (f, F, a, b, description)
cases = [
    (lambda x: 2*x,       lambda x: x**2,        1.0, 3.0,  'int 2x dx = x² | analytic = 8'),
    (lambda x: np.exp(x), lambda x: np.exp(x),   0.0, 1.0,  'int eˣ dx = eˣ | analytic = e-1'),
    (lambda x: 1/x,       lambda x: np.log(x),   1.0, 4.0,  'int 1/x dx = ln|x| | analytic = ln4'),
]

n = 10_000
print(f"{'Description':<45}  {'Analytic':>10}  {'Numerical':>10}  {'Match':>8}")
print('-' * 80)
for f, F, a, b, desc in cases:
    analytic = F(b) - F(a)
    numerical = trapz(f, a, b, n)
    match = abs(analytic - numerical) < 1e-6
    print(f"{desc:<45}  {analytic:>10.6f}  {numerical:>10.6f}  {'✓' if match else '✗':>8}")

## 3. Gaussian Normalization and the 68–95–99.7 Rule

The standard normal PDF is $f(x) = \frac{1}{\sqrt{2\pi}}e^{-x^2/2}$. It must integrate to 1, and the 68–95–99.7 rule gives the probability within $k$ standard deviations.

In [ ]:
normal_pdf = lambda x: np.exp(-x**2 / 2) / np.sqrt(2 * np.pi)

# Normalization (use a large but finite range)
total = trapz(normal_pdf, -8, 8, 100_000)
print(f"∫ N(0,1) dx over [-8, 8] = {total:.10f}  (should be 1.0)")

# 68-95-99.7 rule
print()
for k in [1, 2, 3]:
    prob = trapz(normal_pdf, -k, k, 10_000)
    print(f"P(|X| ≤ {k}σ) = {prob:.4f}  ({prob*100:.2f}%)")

# Visualize
x = np.linspace(-4, 4, 400)
plt.figure(figsize=(9, 4))
plt.plot(x, normal_pdf(x), 'steelblue', lw=2)
colors = ['#cce5ff', '#99ccff', '#66b2ff']
labels = ['68% (±1σ)', '95% (±2σ)', '99.7% (±3σ)']
for k, color, label in zip([3, 2, 1], colors[::-1], labels[::-1]):
    mask = np.abs(x) <= k
    plt.fill_between(x, normal_pdf(x), where=mask, alpha=0.5, color=color, label=label)
plt.xlabel('x')
plt.ylabel('density')
plt.title('Standard normal PDF — 68–95–99.7 rule')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Monte Carlo Integration: Estimating π

A circle of radius 1 inscribed in a $2\times 2$ square has area $\pi$. The fraction of uniform random points in $[-1,1]^2$ that fall inside the circle is $\pi/4$.

$$\pi \approx 4 \cdot \frac{\text{points inside circle}}{\text{total points}}$$

In [ ]:
max_samples = 100_000
pts = rng.uniform(-1, 1, (max_samples, 2))
inside = (pts[:, 0]**2 + pts[:, 1]**2) <= 1.0

# Running estimate
ns = np.logspace(1, 5, 200).astype(int)
estimates = [4 * inside[:n].mean() for n in ns]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Scatter plot of first 5000 points
n_show = 5000
colors = np.where(inside[:n_show], 'steelblue', 'coral')
ax1.scatter(pts[:n_show, 0], pts[:n_show, 1], c=colors, s=1, alpha=0.5)
theta = np.linspace(0, 2*np.pi, 300)
ax1.plot(np.cos(theta), np.sin(theta), 'black', lw=1.5)
ax1.set_aspect('equal')
ax1.set_title(f'Monte Carlo sampling (n={n_show})')

# Convergence
ax2.semilogx(ns, estimates, 'steelblue', lw=1.5, label='estimate')
ax2.axhline(np.pi, color='coral', linestyle='--', lw=2, label=f'π = {np.pi:.5f}')
ax2.set_xlabel('Sample count')
ax2.set_ylabel('Estimate of π')
ax2.set_title('Convergence of Monte Carlo π estimate')
ax2.legend()

plt.tight_layout()
plt.show()

pi_est = 4 * inside.mean()
print(f"Final estimate (n={max_samples}): π ≈ {pi_est:.5f}  (true: {np.pi:.5f})")

## 5. KL Divergence Between Gaussians

$D_{\text{KL}}(p \| q) = \int p(x) \ln\frac{p(x)}{q(x)}\,dx$. We compute this numerically and verify:
- Non-negativity: $D_{\text{KL}} \geq 0$
- Identity: $D_{\text{KL}}(p\|p) = 0$
- Asymmetry: $D_{\text{KL}}(p\|q) \neq D_{\text{KL}}(q\|p)$ in general

In [ ]:
def gaussian_pdf(x, mu, sigma):
    return np.exp(-0.5 * ((x - mu) / sigma)**2) / (sigma * np.sqrt(2 * np.pi))

def kl_numeric(mu_p, sigma_p, mu_q, sigma_q, n=100_000):
    """Numerically integrate D_KL(p||q) for two Gaussians."""
    lo = min(mu_p, mu_q) - 5 * max(sigma_p, sigma_q)
    hi = max(mu_p, mu_q) + 5 * max(sigma_p, sigma_q)
    x = np.linspace(lo, hi, n)
    p = gaussian_pdf(x, mu_p, sigma_p)
    q = gaussian_pdf(x, mu_q, sigma_q)
    # Avoid log(0): only sum where p > 0
    mask = p > 1e-300
    integrand = np.zeros_like(x)
    integrand[mask] = p[mask] * np.log(p[mask] / q[mask])
    return np.trapz(integrand, x)

def kl_analytic(mu_p, sigma_p, mu_q, sigma_q):
    """Closed-form KL divergence between two Gaussians."""
    return (np.log(sigma_q/sigma_p)
            + (sigma_p**2 + (mu_p - mu_q)**2) / (2 * sigma_q**2)
            - 0.5)

cases = [
    (0, 1, 0, 1,   'p=N(0,1), q=N(0,1)  [should be 0]'),
    (0, 1, 2, 1,   'p=N(0,1), q=N(2,1)  [shift by 2σ]'),
    (2, 1, 0, 1,   'p=N(2,1), q=N(0,1)  [reversed: asymmetry]'),
    (0, 1, 0, 2,   'p=N(0,1), q=N(0,2)  [q broader]'),
    (0, 2, 0, 1,   'p=N(0,2), q=N(0,1)  [p broader]'),
]

print(f"{'Case':<42}  {'Numeric':>10}  {'Analytic':>10}")
print('-' * 68)
for mp, sp, mq, sq, desc in cases:
    num = kl_numeric(mp, sp, mq, sq)
    ana = kl_analytic(mp, sp, mq, sq)
    print(f"{desc:<42}  {num:>10.5f}  {ana:>10.5f}")

print()
print("Rows 2 and 3: D_KL(p‖q) ≠ D_KL(q‖p)  — KL divergence is asymmetric.")
print("Row 1: D_KL(p‖p) = 0  — zero divergence from a distribution to itself.")
print("All values ≥ 0  — KL divergence is non-negative.")